# ಪಾಠ 01 - AI ಏಜೆಂಟ್ಗಳನ್ನು ಪರಿಚಯಿಸುವುದು

**AI ಏಜೆಂಟ್ಗಳಿಗಾಗಿ ಆರಂಭಿಕರ ಕೋರ್ಸ್**ನ ಮೊದಲ ಪಾಠಕ್ಕೆ ಸ್ವಾಗತ!

**AI ಏಜೆಂಟ್** ಎಂದರೆ ದೊಡ್ಡ ಭಾಷಾ ಮಾದರಿಯನ್ನು (LLM) ಅದರ ತರ್ಕ ಎಂಜಿನ್ ಆಗಿ ಬಳಸುವ ಪ್ರೋಗ್ರಾಂ ಆಗಿದ್ದು, ಅದು *ಕ್ರಿಯೆಗಳನ್ನು* ನಿಜ ಜಗತ್ತಿನಲ್ಲಿ ತೆಗೆದುಕೊಳ್ಳಬಹುದು — APIಗಳನ್ನು ಕರೆಮಾಡುವುದು, ಡೇಟಾಬೇಸ್‌ಗಳನ್ನು ಪ್ರಶ್ನಿಸುವುದು, ಅಥವಾ ಕೋಡ್ ರನ್ ಮಾಡುವುದು — ಬಳಕೆದಾರರ ಪರವಾಗಿ ಗುರಿಯನ್ನು ಸಾಧಿಸಲು.

ಈ ನೋಟ್‌ಬುಕ್ ನಲ್ಲಿ ನೀವು ನಿಮ್ಮ ಮೊದಲ ಏಜೆಂಟ್ ಅನ್ನು ನಿರ್ಮಿಸಲಿದ್ದೀರಿ: ವಿಮಾನ್ಯ ಶಿಬಿರ ಗಮ್ಯಸ್ಥಳಗಳನ್ನು ಶಿಫಾರಸು ಮಾಡುವ **ಪ್ರಯಾಣ ಏಜೆಂಟ್**. ಪ್ರಯಾಣದಲ್ಲಿದ್ದು ನೀವು ಹೇಗೆ:

1. **Microsoft Agent Framework** ಉಪಯೋಗಿಸಿ Microsoft Foundry Agent Service ನೊಂದಿಗೆ ಸಂಪರ್ಕಿಸುವುದು.
2. ಏಜೆಂಟ್‌ಗೆ ಒಂದು **ಉಪಕರಣ** ನೀಡುವುದು — ಅದು ಕರೆಮಾಡಬಹುದಾದ ಸರಳ Python ಕಾರ್ಯ.
3. ಏಜೆಂಟ್ ಅನ್ನು ಓಡಿಸುವುದು ಮತ್ತು ಅದರ ಪ್ರತಿಕ್ರಿಯೆಯನ್ನು ಪರಿಶೀಲಿಸುವುದು.
4. ಏಜೆಂಟ್‌ದ ಪ್ರತಿಕ್ರಿಯೆಯನ್ನು ಟೋಕನ್‌ತೋ ಟೋಕನ್ ನವಾಗಿ ಪ್ರસારಿಸುವುದು.


## ಸ್ಥಾಪನೆ

ಈ ನೋಟ್ಬುಕ್ ಅನ್ನು ಚಾಲನೆ ಮಾಡುತ್ತಿದ್ದ ಮೊದಲು, ನೀವು ಖಚಿತಪಡಿಸಿಕೊಳ್ಳಿ:

1. **ಮೈಕ್ರೋಸಾಫ್ಟ್ ಫೌಂಡ್ರಿ ಪ್ರಾಜೆಕ್ಟ್** ಜೊತೆಗೆ ನಿಯೋಜಿತ ಚಾಟ್ ಮಾದರಿ (ಉದಾ: `gpt-5-mini`).
2. **ಆಜ್ಯೂರ್ CLI ನಲ್ಲಿ ಲಾಗಿನ್ ಮಾಡಲಾಗಿದೆ** — ನಿಮ್ಮ ಟರ್ಮಿನಲ್‌ನಲ್ಲಿ `az login` ರನ್ ಮಾಡಿ.
3. **ಅಗತ್ಯವಿರುವ ಪರಿಸರ ಚರಗಳನ್ನು ಸಿದ್ಧಪಡಿಸಿ:**
   - `AZURE_AI_PROJECT_ENDPOINT` — ನಿಮ್ಮ ಮೈಕ್ರೋಸಾಫ್ಟ್ ಫೌಂಡ್ರಿ ಪ್ರಾಜೆಕ್ಟ್ ಇಂದಿಂಡು.
   - `AZURE_AI_MODEL_DEPLOYMENT_NAME` — ನಿಮ್ಮ ನಿಯೋಜಿತ ಮಾದರಿಯ ಹೆಸರು.

ಕೆಳಗಿನ ಸೆಲ್ ನೀವು ಅಗತ್ಯವಿರುವ Python ಪ್ಯಾಕೇಜ್‌ಗಳನ್ನು ಸ್ಥಾಪಿಸುತ್ತದೆ.


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import dotenv
from agent_framework.foundry import FoundryChatClient
from azure.identity import AzureCliCredential
from agent_framework import tool

dotenv.load_dotenv(dotenv.find_dotenv())

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
model = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

if not endpoint or not model:
    raise ValueError(
        "Missing required environment variables. "
        "Please set AZURE_AI_PROJECT_ENDPOINT and AZURE_AI_MODEL_DEPLOYMENT_NAME in your .env file."
    )

provider = FoundryChatClient(
    project_endpoint=endpoint,
    model=model,
    credential=AzureCliCredential()
)

## ನಿಮ್ಮ ಮೊದಲ ಏಜೆಂಟ್ ರಚಿಸುವುದು

ಒಂದು ಏಜೆಂಟ್‌ಗೆ ಎರಡು ವಸ್ತುಗಳ ಅಗತ್ಯವಿರುತ್ತದೆ:

- **ನಿರ್ದೇಶಗಳು** ಅದು *ಯಾರು* ಮತ್ತು *ಹೆಗೆ ವರ್ತಿಸಬೇಕು* ಎಂಬುದನ್ನು ಹೇಳುವ (ಸಿಸ್ಟಂ ಪ್ರಾಂಪ್ಟ್).
- **ಉಪಕರಣಗಳು** — Python ಫಂಕ್ಷನ್‌ಗಳು `@tool` ಡೆಕೊರೇಟರ್‌ನೊಂದಿಗೆ, ಏಜೆಂಟ್ ಮಾಹಿತಿ ಪಡೆಯಲು ಅಥವಾ ಕ್ರಿಯೆಗಳು ನಿರ್ವಹಿಸಲು ಕರೆ ಮಾಡಬಹುದು.

ಕೆಳಗೆ ನಾವು ಜನಪ್ರಿಯ ರಜೆ ತಾಣಗಳ ಪಟ್ಟಿ ಹಿಂತಿರುಗಿಸುವ ಸರಳ ಉಪಕರಣವನ್ನು ವ್ಯಾಖ್ಯಾನಿಸಿದ್ದೇವೆ. ಬಳಕೆದಾರರು ಪ್ರಯಾಣ ಶಿಫಾರಸುಗಳನ್ನು ಕೇಳಿದಾಗ ಏಜೆಂಟ್ ಈ ಉಪಕರಣವನ್ನು ಬಳಸುತ್ತದೆ.


In [ ]:
@tool(approval_mode="never_require")
def get_destinations() -> list[str]:
    """Get a list of popular vacation destinations."""
    return [
        "Barcelona",
        "Paris",
        "Berlin",
        "Tokyo",
        "Sydney",
        "New York City",
        "Cairo",
        "Cape Town",
        "Rio de Janeiro",
        "Bali",
    ]

In [ ]:
agent = provider.as_agent(
    name="TravelAgent",
    instructions=(
        "You are a helpful travel agent. Help users find their perfect vacation "
        "destination based on their preferences. Use the get_destinations tool "
        "to see available destinations."
    ),
    tools=[get_destinations],
)

response = await agent.run(
    "I'm looking for a warm beach destination. What do you recommend?"
)
print(response)

## ಸ್ಟ್ರೀಮಿಂಗ್ ಪ್ರತಿಕ್ರಿಯೆಗಳು

ಹೆಚ್ಚು ಸಂವಹನಾತ್ಮಕ ಅನುಭವನ್ನು ನೀಡಲು ನೀವು ಏಜೆಂಟ್‌ನ ಪ್ರತಿಕ್ರಿಯೆಯನ್ನು **ಸ್ಟ್ರೀಮ್** ಮಾಡಬಹುದು. ಸಂಪೂರ್ಣ ಪ್ರತಿಕ್ರಿಯೆಯನ್ನು ಕಾಯುವುದರ ಬದಲು, ಏಜೆಂಟ್ ತಯಾರಾಗುವಂತೆ ತಕ್ಷಣವೇ ಪಠ್ಯದ ತುಂಡುಗಳನ್ನು ನೀಡುತ್ತದೆ. ಇದು ನಿಜಕಾಲದಲ್ಲಿ ಔಟ್‌ಪುಟ್ ಅನ್ನು ತೋರಿಸಬೇಕಾದ ಚಾಟ್ ಇಂಟರ್‌ಫೇಸ್ಗಳಲ್ಲಿ ವಿಶೇಷವಾಗಿ ಉಪಯುಕ್ತವಾಗಿದೆ.


In [ ]:
async for chunk in agent.run(
    "Tell me about Tokyo as a travel destination", stream=True
):
    print(chunk, end="", flush=True)

## ಸಾರಾಂಶ

ಈ ಪಾಠದಲ್ಲಿ ನೀವು ಈ ಕೆಳಗಿನವುಗಳನ್ನು ಕಲಿತುಕೊಂಡಿರಿ:

- **ಪ್ರವ_PROVIDER ರಚಿಸುವುದು** որը `FoundryChatClient` ಮೂಲಕ Microsoft Foundry Agent Service ಗೆ ಸಂಪರ್ಕ ಹೊಂದುತ್ತದೆ.
- **ಉಪಕರಣವನ್ನು** `@tool` ಅಲಂಕರಣೆಯನ್ನು (ಡೇಕೋರೇಟರ್) ಬಳಸಿ ವ್ಯಾಖ್ಯಾನಿಸಿ, ಹೀಗಾಗಿ ಏಜೆಂಟ್ ನಿಖರವಾಗಿ ನಿಮ್ಮ Python ಕಾರ್ಯಗಳನ್ನು ಕರೆದು ಕೊಳ್ಳಬಹುದು.
- **ಏಜೆಂಟ್‍ನನ್ನು ಚಾಲನೆ ಮಾಡಲು** ಬಳಕೆದಾರ ಸಂದೇಶವನ್ನು ನೀಡಿ ಮತ್ತು ಅದರ ಪ್ರತಿಕ್ರಿಯೆಯನ್ನು ಮುದ್ರಿಸಿ.
- **ಪ್ರತಿಕ್ರಿಯೆಗಳ ನದಿ ಹರಿವು** ವಾಸ್ತವಕಾಲದ ಸಮಯದಲ್ಲಿ ಫಲಿತಾಂಶವನ್ನು ಪಡೆಯಲು.

ಮುಂದಿನ ಪಾಠದಲ್ಲಿ ನಾವು ಏಜೆಂಟಿಕ್ ಫ್ರೇಮ್ವರ್ಕ್‌ಗಳನ್ನು ಇನ್ನಷ್ಟು ಗಾಢವಾಗಿ ಪರಿಶೀಲಿಸಿ, ಏಜೆಂಟ್ಗೆ ಹೆಚ್ಚು ಶಕ್ತಿಶಾಲಿ ಸಾಧನಗಳು ಮತ್ತು ಬಹು ಹಂತದ ತರ್ಹೆಯ ಸಾಮರ್ಥ್ಯಗಳನ್ನು ನೀಡುವುದನ್ನು ಕಲಿತುಕೊಳ್ಳುತ್ತೇವೆ.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**ಅಸ್ವೀಕಾರ**:
ಈ ದಸ್ತಾವೇಜು AI ಅನುವಾದ ಸೇವೆ [Co-op Translator](https://github.com/Azure/co-op-translator) ಬಳಸಿ ಅನುವಾದಿಸಲಾಗಿದೆ. ನಾವು ನಿಖರತೆಯನ್ನು ಸಾಧಿಸಲು ಪ್ರಯತ್ನಿಸುತ್ತಿದ್ದರೂ, ದಯವಿಟ್ಟು ಗಮನಿಸಿ, ಸ್ವಯಂಚಾಲಿತ ಅನುವಾದಗಳಲ್ಲಿ ದೋಷಗಳು ಅಥವಾ ಅಸಡ್ಡೆಗಳು ಇರಬಹುದು. ಮೂಲ ಭಾಷೆಯಲ್ಲಿರುವ ಮೂಲ ದಸ್ತಾವೇಜು ಪ್ರಾಮಾಣಿಕ ಮೂಲವೆಂದು ಪರಿಗಣಿಸಬೇಕು. ಪ್ರಮುಖ ಮಾಹಿತಿಗಾಗಿ, ವೃತ್ತಿಪರ ಮಾನವ ಅನುವಾದವನ್ನು ಶಿಫಾರಸು ಮಾಡಲಾಗುತ್ತದೆ. ಈ ಅನುವಾದವನ್ನು ಬಳಸುವ ಮೂಲಕ ಉಂಟಾಗುವ ಯಾವುದೇ ತಪ್ಪು ಅರ್ಥಗಳ ಅಥವಾ ತಪ್ಪು ವ್ಯಾಖ್ಯಾನಗಳ ಬಗ್ಗೆ ನಾವು ಹೊಣೆಗಾರರಲ್ಲ.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
